# Smart Pet Vision release validation
Fresh local CPU kernel with trained weights. This is a regression and service demonstration, not retraining or a new independent test. Historical notebooks remain unchanged.

In [1]:
from pathlib import Path
import json, os, platform, torch
from PIL import Image
from spv.pipeline import Manifest, PetPipeline
torch.set_num_threads(4)
ROOT=Path.cwd()
MANIFEST=Path(os.environ.get('SPV_MANIFEST','runs/manifest.json'))
IMAGES=Path(os.environ.get('SPV_TEAM_IMAGES','data/team/images'))
print(platform.platform(), torch.__version__, 'CPU')
pipeline=PetPipeline(Manifest.load(MANIFEST))
print('All three trained model weights loaded in this fresh kernel')

macOS-27.2-arm64-arm-64bit 2.11.0 CPU


All three trained model weights loaded in this fresh kernel


In [2]:
results=[]
for name in ['team_0005','team_0012','team_0040']:
    result=pipeline.analyze(Image.open(IMAGES/f'{name}.jpg'),False)
    assert result['staff_approval_required'] is True
    assert result['automatic_publication_allowed'] is False
    results.append({'image':name,**result})
    print(name,result['verdict'],result['species'],result['segmentation_matched'])
print('Known mixed-species failure remains visible; no automatic publication is allowed')

team_0005 ready cat True


team_0012 refer_to_staff cat True


team_0040 ready cat True
Known mixed-species failure remains visible; no automatic publication is allowed


In [3]:
import requests
health=requests.get('http://127.0.0.1:8877/health',timeout=120)
health.raise_for_status()
with (IMAGES/'team_0005.jpg').open('rb') as f:
    response=requests.post('http://127.0.0.1:8877/analyze?images=false',files={'file':('team_0005.jpg',f,'image/jpeg')},timeout=120)
response.raise_for_status()
assert response.json()['automatic_publication_allowed'] is False
print('Fresh-kernel HTTP request:',response.status_code)
print('Service response verified in the clean kernel')

Fresh-kernel HTTP request: 200
Service response verified in the clean kernel
